# Chapter 5 &mdash; State Names as Compressed History

**Concept 6 of the Chapter 5 decomposition:** *State Names as Compressed History: "Ends with 0101"*

"Ends with 0101": remember only the longest relevant suffix, and introduce states on demand.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5/Concept-Compressed-History-Suffix/Concept-Compressed-History-Suffix.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


For "**ends with 0101**" the state must remember **the longest prefix of the pattern
that is currently a suffix of the input** &mdash; nothing more. That is compressed history:
the input may be a million bits, the state is one of five.

Two habits make the design mechanical:

* **introduce states on demand** &mdash; add a state the moment you need to distinguish;
* **cover both symbols from every state**, including the fall-back edges, which is
  where beginners lose the machine.

## 2. Definitions

### The specification

In [ ]:
PAT = '0101'
def in_L(s): return s.endswith(PAT)

### State = longest prefix of `0101` that is a suffix of the input so far

In [ ]:
def longest_overlap(s, pat):
    for k in range(min(len(s), len(pat)), -1, -1):
        if s.endswith(pat[:k]):
            return k
    return 0

### The DFA, one state per overlap length, all fall-backs written out

In [ ]:
E = md2mc('''DFA
I    : 0 -> S0      !! have '0'
I    : 1 -> I       !! nothing usable
S0   : 0 -> S0      !! still just '0'
S0   : 1 -> S01     !! have '01'
S01  : 0 -> S010    !! have '010'
S01  : 1 -> I       !! '011' -- start over
S010 : 0 -> S0      !! '0100' -- keep the trailing 0
S010 : 1 -> F       !! '0101' -- match!
F    : 0 -> S010    !! '01010' -- suffix '010' survives
F    : 1 -> I
''')

## 3. Tests

The state name **is** the overlap length &mdash; a checkable invariant.

In [ ]:
tag = {'I': 0, 'S0': 1, 'S01': 2, 'S010': 3, 'F': 4}
from itertools import product
for k in range(11):
    for p in product('01', repeat=k):
        s = ''.join(p)
        assert tag[run_dfa(E, s)] == longest_overlap(s, PAT), s
print("state == longest overlap, verified on all strings up to length 10")

And so the machine is correct.

In [ ]:
assert all(accepts_dfa(E, ''.join(p)) == in_L(''.join(p))
           for k in range(12) for p in product('01', repeat=k))
for s in ['0101', '00101', '01011', '0101 0101'.replace(' ',''), '010', '']:
    print("%-10r ends with 0101? %s" % (s, accepts_dfa(E, s)))

The fall-back edges are the subtle part: `F -0-> S010` keeps the useful suffix.

In [ ]:
print("after '0101' then '0' the input ends '01010'")
print("  longest useful suffix is '010' -> state S010, NOT the start state")
assert run_dfa(E, '01010') == 'S010'
print("Throwing that away would miss '010101'. Check:", accepts_dfa(E, '010101'))
assert accepts_dfa(E, '010101')

Five states, however long the input.

In [ ]:
long_in = '0101' * 150          # 600 symbols
print("|Q| =", len(E["Q"]), " input of length %d handled?" % len(long_in),
      accepts_dfa(E, long_in))
print("(Jove's accepts_dfa recurses per symbol, so Python's recursion limit -- not")
print(" the DFA -- caps the input length. The MACHINE is size 5 either way.)")

## 4. Animation

Trace `010101` and watch the fall-back from `F` land on `S010`, not on `I`.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(E, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Redo it for the pattern `0110`. Which fall-backs differ, and why?
2. For a pattern of length $k$ with no self-overlap, how many states?
3. This construction has a name in string matching. Which algorithm?

In [ ]:
# Your work for the exercises above.